# 10 · Left checks: row check, column check and rollback

This notebook runs the Left half of the distributed protocol (`dtrack_left.checks`) on one real pair. The run folder goes to an S3 scratch prefix, exactly as `dtrack row-check` / `dtrack col-check` would lay it out. It covers `run_row_check`, `public_pair_config`, `make_run_id`, `wait_for_manifests`, `publish_fixed_keys`, `LeftRuntime`, `ColCheckPipeline` and the `rollback` module.

Mirrors `tests/unit/dtrack_left/checks/test_row_check.py`, `tests/unit/dtrack_left/checks/test_col_check.py` and `tests/unit/dtrack_left/checks/test_rollback.py`. Those tests use a memory store and synthetic CSVs. Here everything is real: your Left source, real S3 and (for the Right stand-in) your Databricks SQL warehouse.

## How to approach

**Prerequisites**
- Kernel: the repo `.venv` (`uv sync --extra dbx`, plus `--extra athena` or `--extra ldap` if the pair's Left side needs them).
- The project `.env` (`ENV_FILE`) holds the Left credentials (`<macro>_USR/_PWD/_DSN`, `ORACLE_*`, `LDAP_DSN`). It also holds `DTRACK_DBX_PROFILE` / `DTRACK_DBX_WAREHOUSE_ID`.
- AWS credentials that can write under `S3_SCRATCH`, e.g. `AWS_PROFILE=corp` or the static `corp-winscp` profile from `dtrack dbx s3 creds`.
- A Databricks profile in `~/.databrickscfg` and a SQL warehouse id. You only need these for `RIGHT_MODE = "warehouse"`.

**Run order.** Run it top to bottom. Sections 1-4 only touch the Left source and S3, and they are quick. Section 5 needs the Right side. With `RIGHT_MODE = "warehouse"` the notebook does the Right row check itself through the SQL warehouse. With `"wait"` it waits for notebook 13 or the real Databricks job to do it. Sections 6-8 pull whole windows from the Left source, so they can be slow. `MAX_ROWS` stops the run before an oversized pull. Section 9 (rollback) rewrites only objects under the scratch run.

**What to look at.** Check the per-day counts, that no credential fields appear in the published JSON, the manifest windows, the committed `_INDEX/` pointers and `_STATUS.json` states, and the rollback plans.

**Failure modes**
- `left preflight failed`: wrong credentials, table or `where`, or a missing extra.
- `no active pairs selected`: `PAIR` is skipped or misspelled.
- `TimeoutError` in section 5: nothing released the manifests, because the Right side never ran.
- `Right side preflight failed`: the warehouse cannot see the Right table, or `date_col` is spelled differently.
- `AccessDenied` on S3: the scratch prefix is not writable with the current AWS profile.

Set `KEEP = True` to hand the run folder to notebook 13 (`DTRACK_NB_RUN` = the printed `RUN_URI`).

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# The project .env first, so the DTRACK_* defaults below and the Left credentials resolve
ENV_FILE = os.environ.get("DTRACK_ENV_FILE", "C:/work/dtrack/.env")   # repo .env on this machine
load_dotenv(ENV_FILE, override=False)

CONFIG_PATH = os.environ.get("DTRACK_PROD_CONFIG", "C:/work/dtrack/c.json")   # your real pairs config
PAIR = os.environ.get("DTRACK_NB_PAIR", "orders")             # one active pair of CONFIG_PATH
TO_DATE = None                                                 # None = yesterday; or "YYYY-MM-DD"
DAYS = 3                                                       # window = the last DAYS days up to TO_DATE
S3_SCRATCH = os.environ.get("DTRACK_PROD_S3", "s3://<bucket>/dtrack/nb-scratch")   # the only S3 writes go here
DBX_PROFILE = os.environ.get("DTRACK_DBX_PROFILE", "corp")    # ~/.databrickscfg profile (Right stand-in)
WAREHOUSE_ID = os.environ.get("DTRACK_DBX_WAREHOUSE_ID", "")  # SQL warehouse id (Right stand-in)
DISPOSITION = os.environ.get("DTRACK_LOCAL_DISPOSITION", "external_links")   # inline when presigned downloads are blocked
RIGHT_MODE = "warehouse"                                       # "warehouse" = reconcile here | "wait" = notebook 13 / the job does it
GATE = "auto"                                                  # auto | manual | ratio:<0..1> (Right stand-in only)
WAIT_TIMEOUT = 1800                                            # seconds to wait for manifests in "wait" mode
MAX_ROWS = 200_000                                             # refuse a column check bigger than this many Left rows
APPLY_ROLLBACK = True                                          # section 9 deletes + rebuilds inside the scratch run
LOG_LEVEL = "INFO"                                             # DEBUG shows every SQL statement
WORKDIR = Path.home() / ".local/plans/nbs/_work/10"           # local pulls and copies
KEEP = False                                                   # True = keep the scratch run (for notebook 13)

In [ ]:
import copy
import json
from datetime import date, timedelta

import pandas as pd

from dtrack_left.checks.row_check import make_run_id
from dtrack_left.cli.config import load_config
from dtrack_left.cli.options import setup_logging

# dtrack logs (UTC timestamps) go to stderr, which Jupyter shows under each cell
setup_logging(LOG_LEVEL)

# Load and validate the real config: conn_macro profiles merged, col_map files expanded
config = load_config(CONFIG_PATH)
active = [name for name, row in config["pairs"].items() if not row.get("skip")]
print(f"{len(active)} active pair(s): {active}")
if PAIR not in active:
    raise ValueError(f"PAIR={PAIR!r} is not an active pair of {CONFIG_PATH}")

# The check window: the last DAYS days up to TO_DATE (default yesterday), both ends inclusive
TO = TO_DATE or (date.today() - timedelta(days=1)).isoformat()
FROM = (date.fromisoformat(TO) - timedelta(days=DAYS - 1)).isoformat()

# A deep copy of the pair with its dates narrowed to the window; the real config is never changed
pair = copy.deepcopy(config["pairs"][PAIR])
pair["fromDate"], pair["toDate"] = FROM, TO
run_config = {"pairs": {PAIR: pair}}
DATE_COL = pair["left"]["date_col"]
KEYS = [k.strip() for k in str(pair.get("key") or "").split(",") if k.strip()]
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"pair {PAIR}: {pair['left'].get('source')} -> databricks, window {FROM}..{TO}, key={KEYS or None}")
print(f"col_map: {len(pair['col_map'])} column(s), sample={pair.get('sample')!r}, vintage={pair.get('vintage') or 'day'}")

from dtrack_left.backends.store import S3Store

# One scratch root per run of this notebook; cleanup removes exactly this prefix
RUN_ID = make_run_id(run_config)
NB_ROOT = f"{S3_SCRATCH.rstrip('/')}/nb10-{RUN_ID}"
RUN_URI = f"{NB_ROOT}/runs/{RUN_ID}"
store = S3Store(RUN_URI)
print(f"RUN_ID  = {RUN_ID}\nRUN_URI = {RUN_URI}\nWORKDIR = {WORKDIR}")

## 1. The S3 scratch store

`S3Store` is the only channel between the Left and Right sides. A quick round trip confirms the prefix is writable before any counting. It also shows `exists`, `list_keys` and the `_HEARTBEAT` that the Right side uses to tell whether the Left process is still alive.

In [ ]:
# Write, read back, list and probe one object under a throwaway sub-prefix of the scratch root
probe = S3Store(f"{NB_ROOT}/s3-check")
probe.put_json("probe.json", {"ok": True, "run_id": RUN_ID})
print("get_json  :", probe.get_json("probe.json"))
print("list_keys :", probe.list_keys(""))
print("exists    :", probe.exists("probe.json"), "/ missing:", probe.exists("nope.json"))

# The heartbeat the Right side reads (heartbeat_age) to tell a live Left process from a dead one
probe.heartbeat()
print("heartbeat age (s):", round(probe.heartbeat_age(), 3))
print("bucket / prefix  :", store.bucket, "/", store.location.prefix)

## 2. Run ids and the public pair config

`make_run_id` is a UTC stamp plus a 6-character digest of the config, so the same config in the same second gives the same id. `public_pair_config` is what the Left side publishes for the Right side. It drops every credential and connection field (`SECRET_FIELDS` plus anything ending in `_env`).

In [ ]:
from datetime import UTC, datetime

from dtrack_left.checks.row_check import SECRET_FIELDS, public_pair_config

# Same config + same instant -> same id; a different config -> a different digest
now = datetime.now(UTC)
print("run id          :", make_run_id(run_config, now=now))
print("same again      :", make_run_id(run_config, now=now))
print("whole config    :", make_run_id(config, now=now))

# What leaves this machine for the pair: every side field minus credentials and endpoints
public = public_pair_config(pair)
for side in ("left", "right"):
    dropped = sorted(set(pair[side]) - set(public[side]))
    print(f"{side:5} kept   : {sorted(public[side])}")
    print(f"{side:5} dropped: {dropped}   (SECRET_FIELDS = {sorted(SECRET_FIELDS)} and *_env)")

## 3. Left row check into the scratch run

`run_row_check` preflights the Left source (`SELECT ... WHERE 1=0`) and counts rows per partition in the window. It writes `row_check/<pair>.left.json`, then `row_check/_LEFT_CLOSED` and `_STATUS.json`. This is the same call that `dtrack row-check` makes. It never writes a manifest, because releasing manifests is the Right side's job.

In [ ]:
from dtrack_left.checks.row_check import run_row_check

# Count the real Left source for the window and hand off to the Right side through S3
run_id, documents = run_row_check(run_config, store, run_id=RUN_ID, pair_names=[PAIR], pair_workers=1)
left_doc = store.get_json(f"row_check/{PAIR}.left.json")

# The published document: counts per day, bounds, timing, and no credentials
counts = pd.Series(left_doc["counts"], name="left_rows").sort_index()
print(f"{len(counts)} partition(s), {int(counts.sum())} row(s) in {left_doc['bounds']}, took {left_doc['timing']['left_count_s']}s")
print("preflight   :", left_doc["preflight"])
print("_LEFT_CLOSED:", store.get_json("row_check/_LEFT_CLOSED"))
print("_STATUS.json:", store.get_json("_STATUS.json"))
print("manifests   :", store.list_keys("manifest/") or "none yet (expected: the Right side releases them)")
leaked = [k for side in ("left", "right") for k in left_doc[side] if k in SECRET_FIELDS or k.endswith("_env")]
print("secret fields in the published doc:", leaked or "none")
counts.to_frame()

## 4. How the row check refuses bad input

Two guards from the tests, run on real inputs. Selecting no active pair raises `ValueError` before anything is written. A Left preflight failure raises `RuntimeError` and publishes no counts. The second check uses a deliberately broken copy of your pair, pointed at a table that does not exist. It is a read-only probe against your real source.

In [ ]:
# An empty selection is refused up front
try:
    run_row_check(run_config, S3Store(f"{NB_ROOT}/refused"), pair_names=["no-such-pair"])
except ValueError as exc:
    print("ValueError  :", exc)

# A copy of the pair whose Left relation cannot exist -> preflight fails, no counts are published
broken = copy.deepcopy(run_config)
left = broken["pairs"][PAIR]["left"]
if str(left.get("source")).lower() in {"csv", "file"}:
    left["path"] = str(WORKDIR / "dtrack_missing.csv")
else:
    left["query"] = "SELECT * FROM dtrack_no_such_table_nb10"
refused = S3Store(f"{NB_ROOT}/refused")
try:
    run_row_check(broken, refused, run_id="refused", pair_names=[PAIR])
except RuntimeError as exc:
    print("RuntimeError:", exc)
print("counts published after the failure:", refused.exists(f"row_check/{PAIR}.left.json"))

## 5. Release the manifests (the Right side's row check)

The column check cannot start until the Right side has reconciled the counts and released `manifest/<pair>.json` plus `row_check/_CLOSED`. With `RIGHT_MODE = "warehouse"`, this cell plays the Right side. It calls the real `dtrack_right.checks.row_check.run_row_check` with `dtrack_local`'s `WarehouseTarget`, which reads through your SQL warehouse (read-only). With `"wait"`, run notebook 13 (`DTRACK_NB_RUN = RUN_URI`) or the Databricks job against this run instead.

In [ ]:
from dtrack_left.checks.col_check import wait_for_manifests

# Play the Right side here through the SQL warehouse, or leave it to notebook 13 / the job
if RIGHT_MODE == "warehouse":
    from dtrack_left.backends.databricks import workspace
    from dtrack_local.pipeline import WarehouseTarget
    from dtrack_local.warehouse import Warehouse
    from dtrack_right.checks.row_check import run_row_check as right_row_check

    warehouse = Warehouse(workspace(DBX_PROFILE), WAREHOUSE_ID, disposition=DISPOSITION)
    right_factory = lambda right: WarehouseTarget(warehouse, right, WORKDIR / "right")
    print("right pairs reconciled:", right_row_check(store, right_factory, gate=GATE))
else:
    print(f"waiting up to {WAIT_TIMEOUT}s; point notebook 13 at DTRACK_NB_RUN={RUN_URI}")

# Blocks until row_check/_CLOSED exists; fails fast if the Right side wrote a failed _PREFLIGHT.json
manifests = wait_for_manifests(store, timeout=WAIT_TIMEOUT, poll_interval=5, pair_names=[PAIR])
verdict = store.get_json(f"row_check/{PAIR}.json")
print("summary :", verdict["summary"])
print("gate    :", verdict["gate_decision"])
print("released:", [m["pair"] for m in manifests] or "nothing (gate did not pass; see the partitions below)")
pd.DataFrame(verdict["partitions"]).T

## 6. The manifest, window order and fixed-N sample keys

A manifest groups the matching days into windows (one per `vintage` bucket). Every window × column pair is one unit. `iter_windows_round_robin` interleaves windows across pairs, so one large pair cannot starve the others. When `sample` is an integer N, the Left side picks N keys per window once and shares them as `manifest/<pair>.keys.json` (`publish_fixed_keys`). That needs a key-only pull of each window.

In [ ]:
from dtrack_left.checks.col_check import iter_windows_round_robin, publish_fixed_keys

if not manifests:
    raise RuntimeError("no manifest was released for this pair; nothing to column-check")
manifest = manifests[0]
left_rows = sum(w["n_left"] for w in manifest["windows"])
print(f"sample={manifest['sample']}  vintage={manifest['vintage']}  key_cols={manifest['key_cols']}")
print(f"{len(manifest['windows'])} window(s) x {len(manifest['columns'])} column(s) = {manifest['expected_units']} unit(s), {left_rows} Left row(s)")
print("window order:", [(m["pair"], w["bucket"]) for m, w in iter_windows_round_robin(manifests)])

# Guard: the column check pulls every matching Left row of the window
if left_rows > MAX_ROWS:
    raise RuntimeError(f"{left_rows} Left rows > MAX_ROWS={MAX_ROWS}: lower DAYS or raise MAX_ROWS")

# Fixed-N sampling only: choose and publish the keys once; a second call is a no-op (returns None)
keys_doc = publish_fixed_keys(store, manifest, WORKDIR / "keys")
if keys_doc:
    print("fixed keys per window:", {b: len(v) for b, v in keys_doc["windows"].items()})
    print("published again?     :", publish_fixed_keys(store, manifest, WORKDIR / "keys"))
else:
    print("not a fixed-N sample (or keys already published): nothing to publish")
pd.DataFrame(manifest["windows"]).assign(partitions=lambda f: f["partitions"].map(len))

## 7. One window by hand: LeftRuntime pull and statistics

`LeftRuntime.pull_window` pulls one window (date column, key columns and mapped columns) to a local CSV and applies the sample plan. `compute_stats` then gives the per-column statistics that become the unit payloads. This cell runs that on the first window so you can see the raw CSV and the numbers before the pipeline does it for every window.

In [ ]:
from dtrack_left.checks.col_check import LeftRuntime

# Pull + sample the first window into WORKDIR, then compute the per-column statistics
runtime = LeftRuntime(WORKDIR / "col_check", store)
window = copy.deepcopy(manifest["windows"][0])
sampled = runtime.pull_window(manifest, window)
stats = LeftRuntime.compute_stats(manifest, window, sampled)
print("sampled CSV:", sampled)
display(pd.read_csv(sampled, dtype=object, nrows=5))
pd.DataFrame(stats).T

## 8. The column check pipeline

`ColCheckPipeline.run` pulls windows and computes statistics in two thread pools. For every unit it writes the payload (`units/<id>.json`) before the pointer (`_INDEX/<seq>-<id>.json`). When every expected unit is indexed it writes `manifest/_CLOSED`. A second run publishes nothing because indexed units are skipped. This is the body of `dtrack col-check`.

In [ ]:
from dtrack_left.checks.col_check import ColCheckPipeline

# Publish every unit of every released window; the WIP limit only matters while a Right job drains
pipeline_args = dict(pull_window=runtime.pull_window, compute_stats=runtime.compute_stats, pull_workers=2, stat_workers=2)
published = ColCheckPipeline(store, **pipeline_args).run(manifests)
print("published units      :", published)
print("published on a rerun :", ColCheckPipeline(store, **pipeline_args).run(manifests))
print("manifest/_CLOSED     :", store.get_json("manifest/_CLOSED") if store.exists("manifest/_CLOSED") else "missing (partial)")
print("_STATUS.json         :", store.get_json("_STATUS.json"))
print("ready_unclaimed      :", store.ready_unclaimed())

# The pointers (what the Right side polls) and one payload (what it compares)
pointers = pd.DataFrame([row for _key, row in store.list_json("_INDEX/")])
display(pointers[["sequence", "pair", "window", "column", "right_column", "unit_id"]])
store.get_json(pointers.iloc[0]["payload_key"])

## 9. Rollback on the scratch run

`rollback.plan` lists what must be deleted so a stage is produced again. Commit markers come first, so a crash mid-delete never leaves a stage that looks finished. `rollback.rollback` is a dry run unless `apply=True`. With a config and stage `col`, it rebuilds the manifest from Stage 2 settings only. It refuses when a Stage 1 setting (`left`, `right`, `fromDate`, `toDate`, `date_synonyms`) changed. Every delete here happens inside the scratch run.

In [ ]:
from dtrack_left.checks import rollback

# The three plans (dry): dead = retry dead units, col = redo Stage 2, row = redo Stage 1 and 2
for stage in rollback.STAGES:
    keys = rollback.plan(store, PAIR, stage)
    print(f"{stage:4} {len(keys):4} key(s): {keys[:6]}{' ...' if len(keys) > 6 else ''}")
dry = rollback.rollback(store, PAIR, "col")
print("\ndry run deleted anything?", "deleted" in dry, "| next:", *dry["next"], sep="\n  ")

# Stage 1 guard: moving fromDate is refused for a col rollback (it needs stage 'row')
moved = copy.deepcopy(run_config)
moved["pairs"][PAIR]["fromDate"] = (date.fromisoformat(FROM) + timedelta(days=1)).isoformat()
try:
    rollback.rollback(store, PAIR, "col", config=moved)
except ValueError as exc:
    print("\nValueError:", exc)
try:
    rollback.plan(store, PAIR, "all")
except ValueError as exc:
    print("ValueError:", exc)

In [ ]:
# A Stage 2 change (one mapped column dropped): delete the pair's Stage 2 objects and rebuild its manifest
overrides = set(pair.get("col_type_overrides") or {})
droppable = [c for c in manifest["columns"] if c not in KEYS and c not in overrides]
if not APPLY_ROLLBACK or len(manifest["columns"]) < 2 or not droppable:
    print("skipped: APPLY_ROLLBACK is False or the pair has a single droppable column")
else:
    narrowed = copy.deepcopy(run_config)
    narrowed["pairs"][PAIR]["col_map"].pop(droppable[-1])
    done = rollback.rollback(store, PAIR, "col", config=narrowed, apply=True)
    print(f"dropped {droppable[-1]!r}: deleted {done['deleted']} object(s), new expected_units={done['expected_units']}")
    print("manifest/_CLOSED after rollback:", store.exists("manifest/_CLOSED"))

    # Stage 2 again on the rebuilt manifest, exactly as `dtrack col-check` would after a rollback
    manifests = wait_for_manifests(store, timeout=0, poll_interval=0, pair_names=[PAIR])
    for item in manifests:
        publish_fixed_keys(store, item, WORKDIR / "keys")
    runtime = LeftRuntime(WORKDIR / "col_check_after_rollback", store)
    republished = ColCheckPipeline(store, pull_window=runtime.pull_window, compute_stats=runtime.compute_stats).run(manifests)
    print("republished units:", republished, "| manifest/_CLOSED:", store.get_json("manifest/_CLOSED"))

In [ ]:
import shutil

# The same rollback API on a local run folder: copy the scratch run down and plan against LocalFolder
local_copy = WORKDIR / "run_copy" / RUN_ID
shutil.rmtree(local_copy, ignore_errors=True)
for key in store.list_keys(""):
    target = local_copy / key
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(store.get_bytes(key))
folder = rollback.open_location(str(local_copy))
print(type(folder).__name__, folder.root_uri)
for stage in rollback.STAGES:
    print(f"{stage:4} local {len(rollback.plan(folder, PAIR, stage)):4} key(s) | s3 {len(rollback.plan(store, PAIR, stage)):4} key(s)")

## Cleanup

This deletes every object under this notebook's scratch root (`NB_ROOT`) with the same batched `delete_objects` call the rollback uses, then removes `WORKDIR`. Set `KEEP = True` to keep the run for notebook 13 (`DTRACK_NB_RUN = RUN_URI`).

In [ ]:
import shutil

# Remove exactly what this notebook created: NB_ROOT on S3 and the local WORKDIR
if KEEP:
    print(f"kept {NB_ROOT}\nfor notebook 13: DTRACK_NB_RUN={RUN_URI}")
else:
    scratch = S3Store(NB_ROOT)
    print("deleted", rollback.delete(scratch, scratch.list_keys("")), "object(s) under", NB_ROOT)
    shutil.rmtree(WORKDIR, ignore_errors=True)

## What to check

- Section 1: the scratch round trip works, which proves S3 credentials and prefix permissions.
- Section 3: the per-day Left counts look plausible, `_LEFT_CLOSED` names the pair, and no credential field was published.
- Section 4: an empty selection and a broken source are both refused, and no counts are written for the broken one.
- Section 5: at least one partition is `match`, and the gate passed. Days with `count_diff`, `left_only` or `right_only` are real findings to explain.
- Section 6: the windows follow the pair's `vintage`. For fixed-N sampling, the keys are published once.
- Sections 7-8: `n_total` per column equals the sampled row count, payloads land before pointers, `manifest/_CLOSED` is written, and a rerun publishes 0.
- Section 9: commit markers lead each plan, a `fromDate` change is refused for `col`, and the rebuilt manifest has fewer units.